# grapheme-aware Evaluation Metrics

A **grapheme-aware** evaluation notebook for comparing a hypothesis or model output against reference text using `grapheme_kit.metric`.

These metrics compare grapheme clusters instead of raw Unicode code points, which is important for many writing systems with multi codepoint visible characters.

## Setup


In [5]:
%pip install sacrebleu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Install dependencies
# pip install sacrebleu grapheme_kit

from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
import sys

project_root = Path.cwd()
for candidate in (project_root, project_root.parent, project_root.parent.parent):
    if (candidate / 'src').exists():
        project_root = candidate
        break

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from grapheme_kit.graphemizer import Graphemizer
from grapheme_kit.metric import CER, GraphemeCHRF, charbleu


def quiet_cer(hypothesis: str, reference: str) -> float:
    """Call CER while hiding its current debug print output."""
    with redirect_stdout(StringIO()):
        return CER(hypothesis, reference)


---
## Using Library Metrics

**Reference:** expected or gold text  
**Hypothesis:** predicted text from OCR, ASR, translation, transliteration, or normalization

Use `CER(hypothesis, reference)` for error rate, `GraphemeCHRF().sentence_score(hypothesis, [reference])` for chrF, and `charbleu(reference, hypothesis)` for CharBLEU.

In [7]:
reference = "cat"
hypothesis = "cot"

print(f"Reference : {reference}")
print(f"Hypothesis: {hypothesis}")
print()
print(f"CER(hypothesis, reference) = {quiet_cer(hypothesis, reference):.4f}")
print(f"GraphemeCHRF().sentence_score(...).score = {GraphemeCHRF().sentence_score(hypothesis, [reference]).score:.4f}")
print(f"charbleu(reference, hypothesis) = {charbleu(reference, hypothesis):.4f}")


Reference : cat
Hypothesis: cot

CER(hypothesis, reference) = 0.3333
GraphemeCHRF().sentence_score(...).score = 22.2222
charbleu(reference, hypothesis) = 0.6667


---
## Character Error Rate (CER)

**Operation:** grapheme-aware Levenshtein distance divided by reference grapheme count.  
**Range:** `0.0` is perfect; larger values mean more errors.  
**Best for:** OCR, ASR, spelling correction, and text normalization.

CER is useful when each insertion, deletion, or substitution error should count directly.

In [8]:
pairs = [
    ("MARTHA", "MARHTA"),
    ("DIXON",  "DICKSONX"),
    ("JELLYFISH", "SMELLYFISH"),
    ("abc", "abc"),
    ("abc", "xyz"),
]

print(f"{'reference':<12} {'hypothesis':<12} {'CER':>8}")
print("-" * 36)
for reference, hypothesis in pairs:
    print(f"{reference:<12} {hypothesis:<12} {quiet_cer(hypothesis, reference):>8.4f}")


reference    hypothesis        CER
------------------------------------
MARTHA       MARHTA         0.3333
DIXON        DICKSONX       0.8000
JELLYFISH    SMELLYFISH     0.2222
abc          abc            0.0000
abc          xyz            1.0000


---
## Grapheme chrF

**Compares:** overlapping grapheme n-grams between hypothesis and reference.  
**Range:** `0.0` to `100.0`; higher is better.  
**Best for:** translation, generation, and partial-match evaluation.

chrF gives partial credit when strings share many grapheme sequences, even if the full output is not exact.

In [9]:
chrf = GraphemeCHRF()

pairs = [
    ("MARTHA", "MARHTA"),
    ("DIXON",  "DICKSONX"),
    ("JELLYFISH", "SMELLYFISH"),
    ("Hello வணக்கம் world", "Hello வணக்கம், world!"),
]

print(f"{'reference':<24} {'hypothesis':<24} {'chrF':>8}")
print("-" * 62)
for reference, hypothesis in pairs:
    score = chrf.sentence_score(hypothesis, [reference]).score
    print(f"{reference:<24} {hypothesis:<24} {score:>8.4f}")


reference                hypothesis                   chrF
--------------------------------------------------------------
MARTHA                   MARHTA                    27.5000
DIXON                    DICKSONX                  26.5625
JELLYFISH                SMELLYFISH                80.7972
Hello வணக்கம் world      Hello வணக்கம், world!     75.3096


---
## Grapheme chrF++

**Adds:** word n-grams on top of grapheme n-grams.  
**Range:** `0.0` to `100.0`; higher is better.  
**Best for:** sentence-level output where word order should also matter.

Use `GraphemeCHRF(word_order=2)` when you want chrF++ behavior.

In [10]:
chrf = GraphemeCHRF()
chrf_pp = GraphemeCHRF(word_order=2)

pairs = [
    ("සුබ උදෑසනක්", "සුබ උදෑසනක්"),
    ("මෙය නව මාර්ග වේ", "මෙය නව මාර්ගය වේ"),
    ("இன்று வானிலை மிகவும் அழகாக இருக்கிறது", "இருக்கிறது அழகாக மிகவும் வானிலை இன்று"),
]

print(f"{'reference':<42} {'hypothesis':<42} {'chrF':>8} {'chrF++':>8}")
print("-" * 106)
for reference, hypothesis in pairs:
    score = chrf.sentence_score(hypothesis, [reference]).score
    score_pp = chrf_pp.sentence_score(hypothesis, [reference]).score
    print(f"{reference:<42} {hypothesis:<42} {score:>8.4f} {score_pp:>8.4f}")


reference                                  hypothesis                                     chrF   chrF++
----------------------------------------------------------------------------------------------------------
සුබ උදෑසනක්                                සුබ උදෑසනක්                                100.0000 100.0000
මෙය නව මාර්ග වේ                            මෙය නව මාර්ගය වේ                            78.7749  72.6814
இன்று வானிலை மிகவும் அழகாக இருக்கிறது      இருக்கிறது அழகாக மிகவும் வானிலை இன்று       47.1802  47.8852


---
## CharBLEU

**Compares:** grapheme n-gram precision with a BLEU-style geometric mean.  
**Range:** `0.0` to `1.0`; higher is better.  
**Best for:** compact similarity scores where exact grapheme n-gram matches should be rewarded.

`charbleu(reference, hypothesis)` is useful when you want a BLEU-like score without tokenizing into words.

In [11]:
pairs = [
    ("MARTHA", "MARHTA"),
    ("DIXON",  "DICKSONX"),
    ("JELLYFISH", "SMELLYFISH"),
    ("abc", "abc"),
    ("abc", "xyz"),
]

print(f"{'reference':<12} {'hypothesis':<12} {'CharBLEU':>10}")
print("-" * 38)
for reference, hypothesis in pairs:
    print(f"{reference:<12} {hypothesis:<12} {charbleu(reference, hypothesis):>10.4f}")


reference    hypothesis     CharBLEU
--------------------------------------
MARTHA       MARHTA           0.4642
DIXON        DICKSONX         0.4226
JELLYFISH    SMELLYFISH       0.7598
abc          abc              1.0000
abc          xyz              0.0000


---
## Corpus Evaluation

For a model or dataset, evaluate every hypothesis against its reference, then summarize with mean CER, chrF, chrF++, and CharBLEU.

In [12]:
dataset = [
    ("ab", "ba"),
    ("the", "teh"),
    ("kitten", "sitting"),
]


print(f"{'CER':>8} {'chrF':>8} {'CharBLEU':>10}")
print("-" * 48)

cer_scores = []
charbleu_scores = []
hypotheses = []
references = []

for reference, hypothesis in dataset:
    cer = quiet_cer(hypothesis, reference)
    chrf_score = chrf.sentence_score(hypothesis, [reference]).score
    bleu = charbleu(reference, hypothesis)

    cer_scores.append(cer)
    charbleu_scores.append(bleu)
    hypotheses.append(hypothesis)
    references.append(reference)

    print(f"{cer:>8.4f} {chrf_score:>8.2f} {bleu:>10.4f}")



     CER     chrF   CharBLEU
------------------------------------------------
  1.0000    50.00     1.0000
  0.6667    33.33     1.0000
  0.5000    21.13     0.3365


---
## Choosing the Right Metric

| Metric | Output | Best for |
|--------|--------|----------|
| **CER** | float error rate, lower is better | OCR, ASR, spelling correction, normalization |
| **Grapheme chrF** | `0.0` to `100.0`, higher is better | Partial overlap in generated text |
| **Grapheme chrF++** | `0.0` to `100.0`, higher is better | Generated sentences where word n-grams matter |
| **CharBLEU** | `0.0` to `1.0`, higher is better | BLEU-like grapheme n-gram similarity |

**Key principle:** use CER when you want to count edit errors, and use chrF / chrF++ / CharBLEU when you want a similarity score.